# CWD Oscar Project

## Notebook 03 — Build Oscar Formula Features

This notebook transforms the enriched Best Picture dataset into analysis-ready features for **The Oscar Formula**.

### Input
- `data/processed/best_picture_nominees_enriched.csv`

### Output
- `data/processed/best_picture_formula_features.csv`

In [5]:
!git clone https://github.com/mclary-claritywithdata/cwd-oscar-formula.git
%cd /content/cwd-oscar-formula

Cloning into 'cwd-oscar-formula'...
remote: Enumerating objects: 25, done.
remote: Counting objects: 100% (25/25), done.
remote: Compressing objects: 100% (21/21), done.
remote: Total 25 (delta 5), reused 16 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (25/25), 24.73 KiB | 2.47 MiB/s, done.
Resolving deltas: 100% (5/5), done.
/content/cwd-oscar-formula


In [6]:
import pandas as pd
import numpy as np

In [8]:
df = pd.read_csv("data/processed/best_picture_nominees_enriched.csv")
print("Shape:", df.shape)
display(df.head())

Shape: (650, 38)


,year,film,producers,year_raw,ceremony_year_raw,film_release_year,ceremony_number,winner_flag,film_clean,lookup_year,...,response_flag,error_msg,released_dt,release_month,release_year_omdb,runtime_min,imdb_rating_num,metascore_num,imdb_votes_num,box_office_num
0,1927.0,Wings,"Paramount (Lucien Hubbard, Jesse L. Lasky, B. ...",1927/28 (1st),1927/28 (1st),1927.0,1.0,NaN,Wings,1927.0,...,True,NaN,1929-01-05,1.0,1929.0,144.0,7.5,78.0,15487.0,NaN
1,1927.0,7th Heaven,"Fox (William Fox, producer)",1927/28 (1st),1927/28 (1st),1927.0,1.0,NaN,7th Heaven,1927.0,...,True,NaN,1927-10-30,10.0,1927.0,110.0,7.5,NaN,4348.0,NaN
2,1927.0,The Racket,"The Caddo Company (Howard Hughes, producer)",1927/28 (1st),1927/28 (1st),1927.0,1.0,NaN,The Racket,1927.0,...,False,Movie not found!,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1928.0,The Broadway Melody,Metro-Goldwyn-Mayer (Irving Thalberg & Lawrenc...,1928/29 (2nd) [a],1928/29 (2nd) [a],1928.0,2.0,NaN,The Broadway Melody,1928.0,...,False,Movie not found!,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1928.0,Alibi,"Feature Productions (Roland West, producer)",1928/29 (2nd) [a],1928/29 (2nd) [a],1928.0,2.0,NaN,Alibi,1928.0,...,True,NaN,1928-11-04,11.0,1928.0,NaN,NaN,NaN,NaN,NaN


In [9]:
print(df.columns.tolist())

['year', 'film', 'producers', 'year_raw', 'ceremony_year_raw', 'film_release_year', 'ceremony_number', 'winner_flag', 'film_clean', 'lookup_year', 'omdb_title', 'omdb_year', 'released', 'runtime', 'genre', 'director', 'writer', 'actors', 'plot', 'language', 'country', 'awards', 'poster', 'metascore', 'imdb_rating', 'imdb_votes', 'box_office', 'production', 'response_flag', 'error_msg', 'released_dt', 'release_month', 'release_year_omdb', 'runtime_min', 'imdb_rating_num', 'metascore_num', 'imdb_votes_num', 'box_office_num']


In [10]:
text_cols = ["genre", "director", "actors", "awards"]

for col in text_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()

In [11]:
df["genre_lower"] = df["genre"].astype(str).str.lower()

df["is_drama"] = df["genre_lower"].str.contains("drama", na=False).astype(int)
df["is_biography"] = df["genre_lower"].str.contains("biography", na=False).astype(int)
df["is_history"] = df["genre_lower"].str.contains("history", na=False).astype(int)
df["is_war"] = df["genre_lower"].str.contains("war", na=False).astype(int)
df["is_musical"] = df["genre_lower"].str.contains("musical", na=False).astype(int)
df["is_romance"] = df["genre_lower"].str.contains("romance", na=False).astype(int)
df["is_comedy"] = df["genre_lower"].str.contains("comedy", na=False).astype(int)
df["is_crime"] = df["genre_lower"].str.contains("crime", na=False).astype(int)
df["is_thriller"] = df["genre_lower"].str.contains("thriller", na=False).astype(int)
df["is_horror"] = df["genre_lower"].str.contains("horror", na=False).astype(int)
df["is_scifi"] = df["genre_lower"].str.contains("sci-fi|science fiction", na=False).astype(int)
df["is_fantasy"] = df["genre_lower"].str.contains("fantasy", na=False).astype(int)

In [12]:
df["prestige_genre_flag"] = (
    (
        (df["is_drama"] == 1) |
        (df["is_biography"] == 1) |
        (df["is_history"] == 1) |
        (df["is_war"] == 1)
    )
).astype(int)

In [13]:
df["release_month"] = pd.to_numeric(df["release_month"], errors="coerce")

df["is_q4_release"] = df["release_month"].isin([10, 11, 12]).astype(int)
df["is_december_release"] = (df["release_month"] == 12).astype(int)

In [14]:
df["runtime_min"] = pd.to_numeric(df["runtime_min"], errors="coerce")

df["is_long_runtime"] = (df["runtime_min"] >= 120).astype(int)
df["is_very_long_runtime"] = (df["runtime_min"] >= 140).astype(int)

In [15]:
df["imdb_rating_num"] = pd.to_numeric(df["imdb_rating_num"], errors="coerce")
df["metascore_num"] = pd.to_numeric(df["metascore_num"], errors="coerce")

df["is_high_imdb"] = (df["imdb_rating_num"] >= 7.8).astype(int)
df["is_very_high_imdb"] = (df["imdb_rating_num"] >= 8.2).astype(int)

df["is_high_metascore"] = (df["metascore_num"] >= 80).astype(int)
df["is_very_high_metascore"] = (df["metascore_num"] >= 90).astype(int)

In [16]:
df["box_office_num"] = pd.to_numeric(df["box_office_num"], errors="coerce")

df["is_box_office_50m"] = (df["box_office_num"] >= 50_000_000).astype(int)
df["is_box_office_100m"] = (df["box_office_num"] >= 100_000_000).astype(int)
df["is_box_office_250m"] = (df["box_office_num"] >= 250_000_000).astype(int)

In [17]:
df["awards_lower"] = df["awards"].astype(str).str.lower()

df["mentions_oscar"] = df["awards_lower"].str.contains("oscar", na=False).astype(int)
df["mentions_wins"] = df["awards_lower"].str.contains("win|won", na=False).astype(int)
df["mentions_nominations"] = df["awards_lower"].str.contains("nomination|nominated", na=False).astype(int)

In [18]:
df["oscar_formula_score"] = (
    (2 * df["prestige_genre_flag"]) +
    (2 * df["is_q4_release"]) +
    (1 * df["is_long_runtime"]) +
    (2 * df["is_high_imdb"]) +
    (1 * df["is_high_metascore"]) +
    (1 * df["is_box_office_100m"])
)

In [19]:
def assign_formula_tier(score):
    if pd.isna(score):
        return "Unknown"
    elif score >= 7:
        return "Front-runner"
    elif score >= 4:
        return "Contender"
    else:
        return "Long shot"

df["oscar_formula_tier"] = df["oscar_formula_score"].apply(assign_formula_tier)

In [20]:
qa_cols = [
    "film",
    "genre",
    "release_month",
    "runtime_min",
    "imdb_rating_num",
    "metascore_num",
    "box_office_num",
    "prestige_genre_flag",
    "is_q4_release",
    "is_long_runtime",
    "is_high_imdb",
    "is_high_metascore",
    "is_box_office_100m",
    "oscar_formula_score",
    "oscar_formula_tier"
]

display(df[qa_cols].head(20))

,film,genre,release_month,runtime_min,imdb_rating_num,metascore_num,box_office_num,prestige_genre_flag,is_q4_release,is_long_runtime,is_high_imdb,is_high_metascore,is_box_office_100m,oscar_formula_score,oscar_formula_tier
0,Wings,"Drama, Romance, War",1.0,144.0,7.5,78.0,NaN,1,0,1,0,0,0,3,Long shot
1,7th Heaven,"Drama, Romance",10.0,110.0,7.5,NaN,NaN,1,1,0,0,0,0,4,Contender
2,The Racket,nan,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,0,0,Long shot
3,The Broadway Melody,nan,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,0,0,Long shot
4,Alibi,"Short, Comedy",11.0,NaN,NaN,NaN,NaN,0,1,0,0,0,0,2,Long shot
5,The Hollywood Revue,nan,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,0,0,Long shot
6,In Old Arizona,"Drama, Western",1.0,95.0,5.5,NaN,2834000.0,1,0,0,0,0,0,2,Long shot
7,The Patriot,"Drama, History, Thriller",9.0,113.0,NaN,NaN,NaN,1,0,0,0,0,0,2,Long shot
8,All Quiet on the Western Front,nan,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,0,0,Long shot
9,All Quiet on the Western Front,nan,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,0,0,Long shot


In [21]:
df["oscar_formula_tier"].value_counts(dropna=False)

,count
oscar_formula_tier,
Long shot,464
Contender,152
Front-runner,34


In [22]:
summary = df[[
    "prestige_genre_flag",
    "is_q4_release",
    "is_long_runtime",
    "is_high_imdb",
    "is_high_metascore",
    "is_box_office_100m",
    "oscar_formula_score"
]].mean().sort_values(ascending=False)

summary

,0
oscar_formula_score,2.118462
prestige_genre_flag,0.478462
is_long_runtime,0.255385
is_high_metascore,0.178462
is_q4_release,0.175385
is_high_imdb,0.164615
is_box_office_100m,0.047692


In [23]:
df.to_csv("data/processed/best_picture_formula_features.csv", index=False)

print("Saved: data/processed/best_picture_formula_features.csv")

Saved: data/processed/best_picture_formula_features.csv
